In [1]:
# ==========================================================
# Step 1: Import Required Libraries
# ==========================================================

import pandas as pd
import numpy as np

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [2]:
# ==========================================================
# Step 2: Load AG News Dataset
# ==========================================================

train_df = pd.read_csv(
    r"D:\devhub-advanced\task1-bert-agnews\data\train.csv"
)

test_df = pd.read_csv(
    r"D:\devhub-advanced\task1-bert-agnews\data\test.csv"
)

print("Train Shape:", train_df.shape)
print("Test Shape :", test_df.shape)

Train Shape: (120000, 3)
Test Shape : (7600, 3)


In [3]:
# ==========================================================
# Step 3: Explore Dataset
# ==========================================================

print(train_df.head())

print("\nColumns:")
print(train_df.columns)

print("\nMissing Values:")
print(train_df.isnull().sum())

   Class Index                                              Title  \
0            3  Wall St. Bears Claw Back Into the Black (Reuters)   
1            3  Carlyle Looks Toward Commercial Aerospace (Reu...   
2            3    Oil and Economy Cloud Stocks' Outlook (Reuters)   
3            3  Iraq Halts Oil Exports from Main Southern Pipe...   
4            3  Oil prices soar to all-time record, posing new...   

                                         Description  
0  Reuters - Short-sellers, Wall Street's dwindli...  
1  Reuters - Private investment firm Carlyle Grou...  
2  Reuters - Soaring crude prices plus worries\ab...  
3  Reuters - Authorities have halted oil export\f...  
4  AFP - Tearaway world oil prices, toppling reco...  

Columns:
Index(['Class Index', 'Title', 'Description'], dtype='str')

Missing Values:
Class Index    0
Title          0
Description    0
dtype: int64


In [4]:
# ==========================================================
# Step 4: Rename Columns
# ==========================================================

train_df.rename(columns={
    "Class Index": "label",
    "Title": "title",
    "Description": "description"
}, inplace=True)

test_df.rename(columns={
    "Class Index": "label",
    "Title": "title",
    "Description": "description"
}, inplace=True)

print(train_df.columns)

Index(['label', 'title', 'description'], dtype='str')


In [5]:
# ==========================================================
# Step 5: Combine Title and Description
# ==========================================================

train_df["text"] = train_df["title"] + " " + train_df["description"]
test_df["text"] = test_df["title"] + " " + test_df["description"]

print(train_df[["text", "label"]].head())

                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      3
1  Carlyle Looks Toward Commercial Aerospace (Reu...      3
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      3
3  Iraq Halts Oil Exports from Main Southern Pipe...      3
4  Oil prices soar to all-time record, posing new...      3


In [6]:
# ==========================================================
# Step 6: Convert Labels
# ==========================================================

train_df["label"] = train_df["label"] - 1
test_df["label"] = test_df["label"] - 1

print(train_df["label"].value_counts())

label
2    30000
3    30000
1    30000
0    30000
Name: count, dtype: int64


In [7]:
# ==========================================================
# Step 7: Keep Required Columns
# ==========================================================

train_df = train_df[["text", "label"]]
test_df = test_df[["text", "label"]]

print(train_df.head())
print(train_df.shape)

                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      2
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      2
3  Iraq Halts Oil Exports from Main Southern Pipe...      2
4  Oil prices soar to all-time record, posing new...      2
(120000, 2)


In [8]:
# ==========================================================
# Step 8: Reduce Dataset Size
# ==========================================================

train_df = train_df.sample(
    n=5000,
    random_state=42
).reset_index(drop=True)

test_df = test_df.sample(
    n=1000,
    random_state=42
).reset_index(drop=True)

print(train_df.shape)
print(test_df.shape)

(5000, 2)
(1000, 2)


In [9]:
# ==========================================================
# Step 9: Import Transformers
# ==========================================================

from transformers import AutoTokenizer

print("Transformers Imported Successfully")

D:\devhub-advanced\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers Imported Successfully


In [10]:
# ==========================================================
# Step 10: Load BERT Tokenizer
# ==========================================================

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer Loaded Successfully")


Tokenizer Loaded Successfully


In [11]:
# ==========================================================
# Step 11: Tokenize Dataset
# ==========================================================

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [12]:
# ==========================================================
# Step 12: Convert DataFrame to Dataset
# ==========================================================

from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 5000
})
Dataset({
    features: ['text', 'label'],
    num_rows: 1000
})


In [13]:
# ==========================================================
# Step 13: Apply Tokenization
# ==========================================================

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

print(train_dataset)

Map: 100%|██████████| 1000/1000 [00:00<00:00, 2796.93 examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 5000
})


In [14]:
# ==========================================================
# Step 14: Load BERT Model
# ==========================================================

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

print("BERT Model Loaded Successfully")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERT Model Loaded Successfully


In [15]:
# ==========================================================
# Step 15: Create PyTorch Dataset
# ==========================================================

import torch
from torch.utils.data import Dataset

class AGNewsDataset(Dataset):

    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):

        item = self.dataset[idx]

        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(item["label"], dtype=torch.long)
        }

train_dataset_pt = AGNewsDataset(train_dataset)
test_dataset_pt = AGNewsDataset(test_dataset)

print("Dataset Ready")

Dataset Ready


In [16]:
# ==========================================================
# Step 16: Create DataLoader
# ==========================================================

from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset_pt,
    batch_size=8,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset_pt,
    batch_size=8
)

print("DataLoader Ready")

DataLoader Ready


In [17]:
batch = next(iter(train_loader))

print(batch.keys())
print(batch["input_ids"].shape)
print(batch["labels"].shape)

dict_keys(['input_ids', 'attention_mask', 'labels'])
torch.Size([8, 128])
torch.Size([8])


In [18]:
# ==========================================================
# Step 18: Device and Optimizer
# ==========================================================

import torch
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

print("Device:", device)

Device: cpu


In [19]:
# ==========================================================
# Step 19: Train BERT Model
# ==========================================================

from tqdm.auto import tqdm

model.train()

total_loss = 0

progress_bar = tqdm(train_loader)

for batch in progress_bar:

    optimizer.zero_grad()

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

    loss = outputs.loss

    loss.backward()

    optimizer.step()

    total_loss += loss.item()

    progress_bar.set_description(f"Loss: {loss.item():.4f}")

print("\nTraining Completed")
print("Average Loss:", total_loss / len(train_loader))

Loss: 0.0735: 100%|██████████| 625/625 [1:44:03<00:00,  9.99s/it]  


Training Completed
Average Loss: 0.4007067083030939


In [20]:
# ==========================================================
# Step 20: Evaluate Model
# ==========================================================

from sklearn.metrics import accuracy_score, classification_report

model.eval()

predictions = []
true_labels = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(outputs.logits, dim=1)

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(true_labels, predictions)

print("Accuracy:", accuracy)

print("\nClassification Report:\n")

print(classification_report(true_labels, predictions))

Accuracy: 0.899

Classification Report:

              precision    recall  f1-score   support

           0       0.88      0.93      0.90       236
           1       0.97      0.99      0.98       242
           2       0.92      0.74      0.82       245
           3       0.84      0.94      0.88       277

    accuracy                           0.90      1000
   macro avg       0.90      0.90      0.90      1000
weighted avg       0.90      0.90      0.90      1000



In [21]:
# ==========================================================
# Step 21: Save Fine-Tuned Model
# ==========================================================

model.save_pretrained("./bert_agnews_model")

tokenizer.save_pretrained("./bert_agnews_model")

print("Model Saved Successfully")

Model Saved Successfully


In [22]:
# ==========================================================
# Step 22: Test Prediction
# ==========================================================

label_names = [
    "World",
    "Sports",
    "Business",
    "Sci/Tech"
]

news = "Apple launches a new AI powered iPhone with advanced machine learning features."

inputs = tokenizer(
    news,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)

inputs = {k: v.to(device) for k, v in inputs.items()}

model.eval()

with torch.no_grad():

    outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1).item()

print("News:", news)
print("Predicted Category:", label_names[prediction])

News: Apple launches a new AI powered iPhone with advanced machine learning features.
Predicted Category: Sci/Tech


# ==========================================================
# Step 23: Test Multiple News Headlines
# ==========================================================

news_samples = [
    "NASA discovers a new planet in the solar system.",
    "Pakistan wins the cricket world cup final.",
    "Stock market reaches an all time high today.",
    "Microsoft launches a new AI chatbot."
]

label_names = ["World", "Sports", "Business", "Sci/Tech"]

model.eval()

for news in news_samples:

    inputs = tokenizer(
        news,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()

    print(f"News: {news}")
    print(f"Prediction: {label_names[prediction]}")
    print("-" * 50)

In [ ]:
# Conclusion

In this project, a BERT (bert-base-uncased) model was fine-tuned on the AG News dataset for text classification.

The dataset contains four news categories:
- World
- Sports
- Business
- Sci/Tech

The model was successfully trained and evaluated.

Final Accuracy: **89.9%**

The fine-tuned model can classify unseen news articles into one of the four categories with high accuracy.